# M3 — Optimization Dynamics Laboratory

**An interactive learning laboratory for developing intuition about how optimization works in machine learning and deep learning.**

This notebook guides you to build intuition about:

1. **Optimization as movement** — Parameters moving on a loss landscape
2. **Learning rate** — How step size affects convergence and stability
3. **Curvature and ill-conditioning** — Why gradient descent zig-zags
4. **Momentum** — Smoother descent with accumulated velocity
5. **Adaptive methods** — SGD, Momentum, RMSProp, Adam compared
6. **Saddle points** — Flat regions that slow optimization
7. **Neural network training** — Connecting optimizer dynamics to real models

**Philosophy:** Break complex behavior into observable intermediate steps and visualize everything possible. Every concept includes a visual demonstration.

*NumPy and Matplotlib only for Sections 1–6. Section 7 reuses M2's neural network modules.*

---
## Section 0: Setup

This lab uses **NumPy** and **Matplotlib** only for the core optimization visualizations. Think of optimization as a **physical process** happening in parameter space: we will observe how points (parameters) move across loss landscapes.

Set the random seed for reproducibility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

np.random.seed(42)

# Enable inline plots
%matplotlib inline

---
## Section 1: Optimization as Movement on a Landscape

**Key idea:** Training a model is equivalent to **navigating a loss surface**. Parameters are coordinates; the gradient points in the direction of steepest ascent (we move opposite to it — steepest descent). The path we take is the **optimization trajectory**.

**Concepts:**
- **Loss surface** — The height at each point $(w_1, w_2)$ is the loss $L(w_1, w_2)$
- **Parameters as coordinates** — $(w_1, w_2)$ defines our position in parameter space
- **Gradient** — Points in the direction of steepest *ascent*; we step *opposite* for descent
- **Optimization trajectory** — The sequence of points we visit during training

Below we define a simple 2D convex loss, run gradient descent, and animate the trajectory.

In [ ]:
# Simple 2D convex loss: L(w1, w2) = (w1 - 1)^2 + (w2 - 2)^2
# Minimum at (1, 2)
def loss_quadratic(w):
    return (w[0] - 1)**2 + (w[1] - 2)**2

def grad_quadratic(w):
    return np.array([2 * (w[0] - 1), 2 * (w[1] - 2)])

def gradient_descent(w_init, lr, steps):
    """Vanilla gradient descent."""
    w = w_init.copy()
    trajectory = [w.copy()]
    for _ in range(steps):
        g = grad_quadratic(w)
        w = w - lr * g
        trajectory.append(w.copy())
    return np.array(trajectory)

# Grid for visualization
W1 = np.linspace(-2, 4, 80)
W2 = np.linspace(-1, 5, 80)
WW1, WW2 = np.meshgrid(W1, W2)
L_grid = (WW1 - 1)**2 + (WW2 - 2)**2

# Run gradient descent from a random starting point
w_init = np.array([3.5, -0.5])
traj = gradient_descent(w_init, lr=0.15, steps=40)

In [ ]:
# Visualize: 3D surface and contour plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(WW1, WW2, L_grid, cmap='viridis', alpha=0.8)
ax1.set_xlabel('$w_1$')
ax1.set_ylabel('$w_2$')
ax1.set_zlabel('Loss')
ax1.set_title('Loss Surface')

# Contour + trajectory
ax2 = fig.add_subplot(122)
ax2.contourf(WW1, WW2, L_grid, levels=20, cmap='viridis', alpha=0.6)
ax2.contour(WW1, WW2, L_grid, levels=15, colors='black', alpha=0.3, linewidths=0.5)
ax2.plot(traj[:, 0], traj[:, 1], 'r-o', markersize=4, linewidth=1.5, label='Trajectory')
ax2.plot(traj[0, 0], traj[0, 1], 'go', markersize=10, label='Start')
ax2.plot(traj[-1, 0], traj[-1, 1], 'b*', markersize=14, label='End')
ax2.set_xlabel('$w_1$')
ax2.set_ylabel('$w_2$')
ax2.set_title('Optimization Trajectory on Contours')
ax2.legend()
ax2.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Animation: trajectory building step by step (observe optimization as movement)
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(WW1, WW2, L_grid, levels=20, cmap='viridis', alpha=0.6)
ax.contour(WW1, WW2, L_grid, levels=15, colors='black', alpha=0.3, linewidths=0.5)

line, = ax.plot([], [], 'r-o', markersize=5, linewidth=1.5)

def init():
    line.set_data([], [])
    return [line]

def update(frame):
    line.set_data(traj[:frame+1, 0], traj[:frame+1, 1])
    return [line]

anim = FuncAnimation(fig, update, init_func=init, frames=len(traj), interval=80, blit=True)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Animated: Optimization trajectory — watch the point move toward the minimum')
ax.set_aspect('equal')
plt.tight_layout()
HTML(anim.to_jshtml())

**Reflection:** What did you notice about how the point moves relative to the contour lines? The trajectory crosses contours perpendicularly in the descent direction — this is because the gradient is perpendicular to contour lines (level curves).

---
## Section 2: Learning Rate and Optimization Behavior

**Key idea:** The **learning rate** ($\eta$) controls the step size in parameter space. Too small → slow convergence. Too large → overshooting, oscillation, or divergence.

**Concepts:**
- **Small learning rate** — Tiny steps; many iterations to reach the minimum; safe but slow
- **Appropriate learning rate** — Efficient convergence without overshooting
- **Large learning rate** — Big steps; may overshoot and oscillate around the minimum
- **Very large learning rate** — Can diverge (loss increases) or oscillate wildly

Below we compare trajectories under different learning rates on the same loss surface.

In [ ]:
# Same quadratic loss and grid
lrs = [0.01, 0.15, 0.8, 1.5]
labels = ['LR=0.01 (small)', 'LR=0.15 (good)', 'LR=0.8 (large)', 'LR=1.5 (too large)']
colors = ['blue', 'green', 'orange', 'red']
trajectories = []
loss_curves = []

for lr, color in zip(lrs, colors):
    traj = gradient_descent(w_init, lr=lr, steps=30)
    trajectories.append(traj)
    losses = [loss_quadratic(w) for w in traj]
    loss_curves.append(losses)

In [ ]:
# Animated visualization: show all GD trajectories and loss curves for all LRs in a single sequential animation

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Use a consistent min loss for all plots for comparability
all_min_loss = min(min(losses) for losses in loss_curves)

def make_anim_for_all_lrs(trajectories, loss_curves, colors, labels, lrs, min_loss=None):
    num_lrs = len(lrs)
    max_steps = max(len(traj) for traj in trajectories)
    # Each phase is one learning rate; let's animate them in sequence with their respective length
    phase_lengths = [len(traj) for traj in trajectories]
    total_frames = sum(phase_lengths)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax1, ax2 = axes

    # Plot loss surface in first subplot (fixed for all LRs)
    ax1.contourf(WW1, WW2, L_grid, levels=20, cmap='viridis', alpha=0.6)
    ax1.contour(WW1, WW2, L_grid, levels=15, colors='black', alpha=0.3, linewidths=0.5)
    ax1.plot(1, 2, 'k*', markersize=14, label='Minimum')
    ax1.set_xlabel('$w_1$')
    ax1.set_ylabel('$w_2$')
    ax1.set_aspect('equal')

    # Plot for each LR but initially invisible
    lines_traj = []
    for color, label in zip(colors, labels):
        line_traj, = ax1.plot([], [], '-o', color=color, markersize=3, linewidth=2, label=label)
        lines_traj.append(line_traj)
    ax1.legend(loc='upper right', fontsize=10)

    # Loss vs step in second subplot
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Loss')
    ax2.set_yscale('symlog', linthresh=0.01)
    ax2.set_xlim(0, max_steps)
    # Extend minimum y limit for better view (make it 10% lower than min loss, but min zero)
    if min_loss is None:
        min_loss_val = min(min(losses) for losses in loss_curves)
    else:
        min_loss_val = min_loss
    y_min = max(0, min_loss_val - 0.1 * abs(min_loss_val))
    y_max = max(max(losses) for losses in loss_curves)
    ax2.set_ylim(y_min, y_max)
    lines_loss = []
    for color, label in zip(colors, labels):
        line_loss, = ax2.plot([], [], color=color, linewidth=2, label=label)
        lines_loss.append(line_loss)
    ax2.legend()

    # Helper to map a global frame idx to which phase (LR) & which step within that phase
    phase_starts = [0]
    for plen in phase_lengths[:-1]:
        phase_starts.append(phase_starts[-1] + plen)
    def find_current_phase(frame_idx):
        for phase, start in enumerate(phase_starts):
            if frame_idx < start + phase_lengths[phase]:
                return phase, frame_idx - start
        return num_lrs-1, phase_lengths[-1]-1

    def init():
        # Clear all lines
        for line_traj in lines_traj:
            line_traj.set_data([], [])
        for line_loss in lines_loss:
            line_loss.set_data([], [])
        ax1.set_title('') 
        ax2.set_title('')
        return lines_traj + lines_loss

    def animate(frame_idx):
        phase, frame_in_phase = find_current_phase(frame_idx)
        # Update the current phase trajectory/loss, leave others empty
        for i in range(num_lrs):
            if i == phase:
                traj = trajectories[i]
                losses = loss_curves[i]
                idx = min(frame_in_phase, len(traj)-1, len(losses)-1)
                lines_traj[i].set_data(traj[:idx+1, 0], traj[:idx+1, 1])
                lines_loss[i].set_data(range(idx+1), losses[:idx+1])
            else:
                lines_traj[i].set_data([], [])
                lines_loss[i].set_data([], [])
        ax1.set_title(f'GD Trajectory ({labels[phase]})', color=colors[phase])
        ax2.set_title(f'Loss vs Step ({labels[phase]})', color=colors[phase])
        return lines_traj + lines_loss

    anim = FuncAnimation(fig, animate, frames=total_frames, init_func=init, interval=100, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

# Show a single animation for all learning rates, animating one after the other
display(make_anim_for_all_lrs(trajectories, loss_curves, colors, labels, lrs, min_loss=all_min_loss))

**Reflection:** How does step size affect convergence speed and stability? Notice how the small LR creeps slowly; the good LR converges cleanly; the large LRs overshoot and oscillate (or diverge). The learning rate is the single most important hyperparameter for gradient descent.

---
## Section 3: Curvature and Ill-Conditioning

**Key idea:** Not all directions in the loss landscape behave the same. In an **elliptical** (elongated) valley, curvature is high in one direction and low in another. Gradient descent takes small steps in steep directions and large steps in shallow directions — but the gradient points mainly along the steep direction, causing a **zig-zag** path.

**Concepts:**
- **Curvature** — How sharply the loss bends in each direction
- **Narrow vs wide valleys** — Elliptical contours mean different curvatures per axis
- **Condition number** — Ratio of largest to smallest curvature; high = ill-conditioned
- **Zig-zag motion** — Gradient descent overshoots in the shallow direction, bounces between valley walls

Below we use an elliptical loss $L(w_1, w_2) = 100 (w_1 - 1)^2 + (w_2 - 2)^2$ to create a narrow valley.

In [ ]:
# Elliptical loss: L = a*(w1-1)^2 + b*(w2-2)^2  with a >> b
# Creates a narrow valley along w2 axis
a, b = 100, 1
def loss_elliptical(w):
    return a * (w[0] - 1)**2 + b * (w[1] - 2)**2

def grad_elliptical(w):
    return np.array([2*a*(w[0]-1), 2*b*(w[1]-2)])

def gd_elliptical(w_init, lr, steps):
    w = w_init.copy()
    traj = [w.copy()]
    for _ in range(steps):
        g = grad_elliptical(w)
        w = w - lr * g
        traj.append(w.copy())
    return np.array(traj)

# Condition number = a/b = 100 (ill-conditioned)
print(f"Condition number (a/b) = {a/b:.0f} — high ratio means narrow valley, gradient descent zig-zags")

# Grid and trajectory
W1e = np.linspace(-1, 3, 100)
W2e = np.linspace(-2, 6, 100)
We1, We2 = np.meshgrid(W1e, W2e)
L_ellip = a * (We1 - 1)**2 + b * (We2 - 2)**2

w_init_e = np.array([0.5, -1.0])
traj_zigzag = gd_elliptical(w_init_e, lr=0.01, steps=80)

In [ ]:
# Contour plot: elongated ellipses + zig-zag trajectory
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(We1, We2, L_ellip, levels=30, cmap='viridis', alpha=0.6)
ax.contour(We1, We2, L_ellip, levels=20, colors='black', alpha=0.3, linewidths=0.5)
ax.plot(traj_zigzag[:, 0], traj_zigzag[:, 1], 'r-o', markersize=3, linewidth=1)
ax.plot(1, 2, 'b*', markersize=14, label='Minimum')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Zig-Zag: Gradient Descent in an Ill-Conditioned Valley')
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.show()

**Reflection:** Why does the optimizer bounce between valley walls instead of going straight down? The gradient points mostly in the steep $w_1$ direction; each step overshoots in the shallow $w_2$ direction. The result is a zig-zag. Momentum and adaptive methods help smooth this out.

---
## Section 4: Momentum and Stabilized Descent

**Key idea:** **Momentum** adds inertia: instead of stepping only with the current gradient, we accumulate a *velocity* $v$ and use it to smooth oscillations. The update is $v_{t+1} = \beta v_t + \nabla L$, then $\theta_{t+1} = \theta_t - \eta v_{t+1}$.

**Concepts:**
- **Oscillation reduction** — Velocity in one direction partially cancels velocity in the opposite direction
- **Accumulated velocity** — Past gradients contribute to the current step
- **Smoother trajectories** — Momentum helps "roll" down shallow valleys instead of zig-zagging

Below we compare vanilla SGD vs momentum ($\beta = 0.9$) on the same elliptical loss.

In [ ]:
def momentum_descent(w_init, lr, beta, steps, loss_fn, grad_fn):
    """Momentum: v = beta*v + grad, w = w - lr*v"""
    w = w_init.copy()
    v = np.zeros_like(w)
    traj = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        v = beta * v + g
        w = w - lr * v
        traj.append(w.copy())
    return np.array(traj)

# Compare SGD vs Momentum on elliptical loss
traj_sgd = gd_elliptical(w_init_e, lr=0.01, steps=50)
traj_mom = momentum_descent(w_init_e, lr=0.01, beta=0.9, steps=50, loss_fn=loss_elliptical, grad_fn=grad_elliptical)

In [ ]:
# Overlaid trajectories: SGD (zig-zag) vs Momentum (smoother)
fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(We1, We2, L_ellip, levels=30, cmap='viridis', alpha=0.6)
ax.contour(We1, We2, L_ellip, levels=20, colors='black', alpha=0.3, linewidths=0.5)
ax.plot(traj_sgd[:, 0], traj_sgd[:, 1], 'r-o', markersize=2, linewidth=1, label='SGD', alpha=0.8)
ax.plot(traj_mom[:, 0], traj_mom[:, 1], 'c-s', markersize=2, linewidth=1, label='Momentum (β=0.9)', alpha=0.8)
ax.plot(1, 2, 'b*', markersize=14, label='Minimum')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('SGD vs Momentum: Momentum smooths oscillations')
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3D animation: overlay SGD and Momentum trajectories on the loss surface
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(We1, We2, L_ellip, cmap='viridis', alpha=0.8, rstride=2, cstride=2)

# Prepare SGD and Momentum 3D trajectory lines
line_sgd, = ax.plot([], [], [], 'r-o', markersize=4, linewidth=2, label='SGD')
line_mom, = ax.plot([], [], [], 'c-s', markersize=4, linewidth=2, label='Momentum (β=0.9)')

ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_zlabel('Loss')
ax.set_title('Animated: Convergence of SGD vs Momentum on ill-conditioned surface')
ax.legend()

z_traj_sgd = np.array([loss_elliptical(w) for w in traj_sgd])
z_traj_mom = np.array([loss_elliptical(w) for w in traj_mom])

def init_3d_both():
    line_sgd.set_data([], [])
    line_sgd.set_3d_properties([])
    line_mom.set_data([], [])
    line_mom.set_3d_properties([])
    return [line_sgd, line_mom]

def update_3d_both(frame):
    # Draw as far as each trajectory is available
    idx_sgd = min(frame, len(traj_sgd)-1)
    idx_mom = min(frame, len(traj_mom)-1)
    line_sgd.set_data(traj_sgd[:idx_sgd+1, 0], traj_sgd[:idx_sgd+1, 1])
    line_sgd.set_3d_properties(z_traj_sgd[:idx_sgd+1])
    line_mom.set_data(traj_mom[:idx_mom+1, 0], traj_mom[:idx_mom+1, 1])
    line_mom.set_3d_properties(z_traj_mom[:idx_mom+1])
    return [line_sgd, line_mom]

max_frames = max(len(traj_sgd), len(traj_mom))
anim_3d_both = FuncAnimation(fig, update_3d_both, init_func=init_3d_both, frames=max_frames, interval=60, blit=False)
# plt.tight_layout()
HTML(anim_3d_both.to_jshtml())

**Reflection:** How does momentum change the path compared to vanilla gradient descent? Momentum builds up speed in the direction of consistent gradients (down the valley) and dampens oscillations across the valley. The result is a more direct path to the minimum.

---
## Section 5: Adaptive Optimization Methods

**Key idea:** **Adaptive** methods use a *per-parameter* learning rate, scaling updates by the history of gradients. In ill-conditioned landscapes, this can speed up progress along shallow directions.

**Concepts:**
- **Per-parameter learning rate** — Each parameter gets its own effective step size
- **Scaling by gradient history** — RMSProp uses squared-gradient moving average; Adam combines momentum + RMSProp
- **Faster convergence** — Adaptive methods often reach the minimum in fewer steps on difficult landscapes

**Optimizers compared:**
- **SGD** — $\theta \leftarrow \theta - \eta \nabla L$
- **Momentum** — $v \leftarrow \beta v + \nabla L$, $\theta \leftarrow \theta - \eta v$
- **RMSProp** — $g^2 \leftarrow \rho g^2 + (1-\rho)(\nabla L)^2$, $\theta \leftarrow \theta - \eta \nabla L / \sqrt{g^2 + \epsilon}$
- **Adam** — Combines momentum (first moment) + RMSProp-like second moment, with bias correction

In [ ]:
def rmsprop_descent(w_init, lr, rho, eps, steps, grad_fn):
    """RMSProp: g2 = rho*g2 + (1-rho)*g^2, w = w - lr*g/sqrt(g2+eps)"""
    w = w_init.copy()
    g2 = np.zeros_like(w)
    traj = [w.copy()]
    for _ in range(steps):
        g = grad_fn(w)
        g2 = rho * g2 + (1 - rho) * g**2
        w = w - lr * g / (np.sqrt(g2) + eps)
        traj.append(w.copy())
    return np.array(traj)

def adam_descent(w_init, lr, beta1, beta2, eps, steps, grad_fn):
    """Adam: m = beta1*m + (1-beta1)*g, v = beta2*v + (1-beta2)*g^2, bias-correct, w = w - lr*m/sqrt(v)+eps"""
    w = w_init.copy()
    m, v = np.zeros_like(w), np.zeros_like(w)
    traj = [w.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(w.copy())
    return np.array(traj)

# Run all four optimizers
steps = 50
traj_sgd5 = gd_elliptical(w_init_e, lr=0.01, steps=steps)
traj_mom5 = momentum_descent(w_init_e, lr=0.01, beta=0.9, steps=steps, loss_fn=loss_elliptical, grad_fn=grad_elliptical)
traj_rms = rmsprop_descent(w_init_e, lr=0.1, rho=0.9, eps=1e-8, steps=steps, grad_fn=grad_elliptical)
traj_adam = adam_descent(w_init_e, lr=0.3, beta1=0.9, beta2=0.999, eps=1e-8, steps=steps, grad_fn=grad_elliptical)

In [ ]:
# Four trajectories + loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trajs = [traj_sgd5, traj_mom5, traj_rms, traj_adam]
names = ['SGD', 'Momentum', 'RMSProp', 'Adam']
clrs = ['red', 'cyan', 'orange', 'magenta']

ax1 = axes[0]
ax1.contourf(We1, We2, L_ellip, levels=30, cmap='viridis', alpha=0.6)
ax1.contour(We1, We2, L_ellip, levels=20, colors='black', alpha=0.3, linewidths=0.5)
for traj, name, c in zip(trajs, names, clrs):
    ax1.plot(traj[:, 0], traj[:, 1], '-o', color=c, markersize=1.5, linewidth=0.8, label=name)
ax1.plot(1, 2, 'b*', markersize=14, label='Minimum')
ax1.set_xlabel('$w_1$')
ax1.set_ylabel('$w_2$')
ax1.set_title('Optimizer Trajectories on Ill-Conditioned Loss')
ax1.legend()
ax1.set_aspect('equal')

ax2 = axes[1]
for traj, name, c in zip(trajs, names, clrs):
    losses = [loss_elliptical(w) for w in traj]
    ax2.plot(losses, color=c, label=name)
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')
ax2.set_title('Loss vs Step')
ax2.legend()
ax2.set_yscale('log')
plt.tight_layout()
plt.show()

**Reflection:** Which optimizer reaches the minimum fastest? How do their paths differ? RMSProp and Adam adapt the step size per dimension — they typically make faster progress along the shallow direction of the valley. SGD zig-zags; Momentum smooths but still uses a single global learning rate.

---
## Section 6: Saddle Points in High-Dimensional Optimization

**Key idea:** A **saddle point** is where the loss curves *up* in some directions and *down* in others. The gradient can be zero or very small there. In high-dimensional neural networks, saddle points are common; local minima are rarer. Optimization can slow down significantly near saddles.

**Concepts:**
- **Saddle point** — A critical point that is a minimum in some directions, maximum in others
- **Flat regions** — Near the saddle, gradients are small; progress is slow
- **Slow escape** — Gradient descent may take many steps to move away from a saddle

Below we use $L(w_1, w_2) = w_1^2 - w_2^2$ — a classic saddle at the origin. (No global minimum; we observe the geometry and slow progress near the saddle.)

In [ ]:
# Saddle: L = w1^2 - w2^2  (saddle at origin; no global min, loss -> -inf as w2 -> ±inf)
def loss_saddle(w):
    return w[0]**2 - w[1]**2

def grad_saddle(w):
    return np.array([2*w[0], -2*w[1]])

def gd_saddle(w_init, lr, steps):
    w = w_init.copy()
    traj = [w.copy()]
    for _ in range(steps):
        g = grad_saddle(w)
        w = w - lr * g
        traj.append(w.copy())
    return np.array(traj)

# Grid for saddle
Ws1 = np.linspace(-2, 2, 80)
Ws2 = np.linspace(-2, 2, 80)
Ws1g, Ws2g = np.meshgrid(Ws1, Ws2)
L_saddle = Ws1g**2 - Ws2g**2

# Start near saddle (0.1, 0.1) - gradient is small, progress is slow
w_saddle_init = np.array([0.5, 0.5])
traj_saddle = gd_saddle(w_saddle_init, lr=0.1, steps=30)

In [ ]:
# 3D interactive animation: saddle surface with GD trajectory
import plotly.graph_objs as go
import numpy as np

# Compute trajectory values
x_traj = traj_saddle[:, 0]
y_traj = traj_saddle[:, 1]
z_traj = np.array([loss_saddle([x, y]) for x, y in zip(x_traj, y_traj)])

# Saddle surface
surface = go.Surface(
    x=Ws1g,
    y=Ws2g,
    z=L_saddle,
    opacity=0.7,
    colorscale='Viridis',
    showscale=False,
    name="Saddle"
)

# Animation frames — show trajectory line growing
frames = [
    go.Frame(
        data=[
            # The surface stays static
            surface,
            # Red trajectory up to frame i
            go.Scatter3d(
                x=x_traj[:i+1],
                y=y_traj[:i+1],
                z=z_traj[:i+1],
                mode='lines+markers',
                line=dict(color='red', width=5),
                marker=dict(size=5),
                name="Trajectory"
            ),
            # The saddle point marker at origin
            go.Scatter3d(
                x=[0], y=[0], z=[loss_saddle([0, 0])],
                mode='markers',
                marker=dict(color='black', size=10, symbol='x'),
                name="Saddle (grad=0)"
            )
        ]
    ) for i in range(len(traj_saddle))
]

# Initial data is surface + just the starting point
init_trajectory = go.Scatter3d(
    x=[x_traj[0]],
    y=[y_traj[0]],
    z=[z_traj[0]],
    mode='lines+markers',
    line=dict(color='red', width=5),
    marker=dict(size=5),
    name="Trajectory"
)
saddle_marker = go.Scatter3d(
    x=[0],
    y=[0],
    z=[loss_saddle([0, 0])],
    mode='markers',
    marker=dict(color='black', size=10, symbol='x'),
    name="Saddle (grad=0)"
)
layout = go.Layout(
    title='Saddle Surface and GD Trajectory (interactive 3D)',
    scene=dict(
        xaxis_title='$w_1$',
        yaxis_title='$w_2$',
        zaxis_title='Loss',
        camera=dict(eye=dict(x=1.5, y=1.6, z=0.75)),
    ),
    width=900,
    height=500,
    legend=dict(x=0.02, y=0.98),
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(label="Play",
                     method="animate",
                     args=[None, {"frame": {"duration": 120, "redraw": True}, "fromcurrent": True, "transition": {"duration": 0}}]),
                dict(label="Pause",
                     method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}])
            ],
            showactive=False,
        )
    ]
)

fig_inter = go.Figure(
    data=[surface, init_trajectory, saddle_marker],
    layout=layout,
    frames=frames
)
fig_inter.show()

In [ ]:
# Loss along trajectory — observe slow progress when passing through saddle region
losses_saddle = [loss_saddle(w) for w in traj_saddle]
plt.figure(figsize=(7, 4))
plt.plot(losses_saddle, 'r-o', markersize=4)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Loss Along Trajectory: Saddle creates flat/slow region')
plt.tight_layout()
plt.show()

**Reflection:** Why might an optimizer get stuck or move slowly near a saddle? At the saddle, the gradient is zero — we would be stuck. Slightly off the saddle, gradients are small, so steps are tiny. In high-dimensional neural networks, many directions can be flat (zero or near-zero curvature), creating saddle-like regions that slow training. Momentum can sometimes help escape saddles by carrying velocity through flat regions.

---
## Section 7: Optimization in Neural Networks

**Key idea:** Connect the optimizer dynamics we observed in 2D to **real neural network training**. We reuse M2's `MultiLayerNN` and run a custom training loop with different optimizers (SGD, Momentum, Adam) to compare loss curves.

**Concepts:**
- **Loss landscapes in NNs** — High-dimensional, non-convex; contain many saddle points
- **Gradient flow** — Backpropagation computes gradients; optimizers determine how we update
- **Training instability** — Poor optimizers or learning rates lead to slow or unstable training

We use `make_moons` data and a small MLP `[2, 10, 1]` with ReLU hidden and sigmoid output.

In [ ]:
# Add M2 directory to path and import MultiLayerNN
import sys
import os

m2_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), "M2_neural_network_from_scratch"))
if m2_path not in sys.path:
    sys.path.insert(0, m2_path)

from multi_layer_nn import MultiLayerNN
from sklearn.datasets import make_moons

# Generate data
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)
y_moons_2d = y_moons.reshape(-1, 1)
print("Moons data:", X_moons.shape, y_moons.shape)

In [ ]:
# Custom training loop with SGD, Momentum, or Adam
# Returns loss history. Creates a COPY of initial weights for fair comparison.

def create_nn(seed=42):
    """Fresh 2-layer MLP: [2, 10, 1], ReLU + sigmoid."""
    layers = [(2, 10, "relu"), (10, 1, "sigmoid")]
    return MultiLayerNN(layers, initialization="he", seed=seed)

def train_sgd(nn, X, y, epochs, lr=0.05):
    nn.learning_rate = lr
    loss_hist = []
    for _ in range(epochs):
        caches = nn.forward(X)
        y_hat = caches[-1][1]
        loss = nn.compute_loss(y, y_hat)
        loss_hist.append(loss)
        wGrads, bGrads, _ = nn.backward(X, y, caches)
        nn.update_params(wGrads, bGrads)
    return loss_hist

def train_momentum(nn, X, y, epochs, lr=0.05, beta=0.9):
    m_W = [np.zeros_like(l.weights) for l in nn.layers]
    m_b = [np.zeros_like(l.biases) for l in nn.layers]
    loss_hist = []
    for _ in range(epochs):
        caches = nn.forward(X)
        y_hat = caches[-1][1]
        loss = nn.compute_loss(y, y_hat)
        loss_hist.append(loss)
        wGrads, bGrads, _ = nn.backward(X, y, caches)
        for i, layer in enumerate(nn.layers):
            m_W[i] = beta * m_W[i] + wGrads[i]
            m_b[i] = beta * m_b[i] + bGrads[i]
            layer.weights -= lr * m_W[i]
            layer.biases -= lr * m_b[i]
    return loss_hist

def train_adam(nn, X, y, epochs, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    m_W = [np.zeros_like(l.weights) for l in nn.layers]
    m_b = [np.zeros_like(l.biases) for l in nn.layers]
    v_W = [np.zeros_like(l.weights) for l in nn.layers]
    v_b = [np.zeros_like(l.biases) for l in nn.layers]
    loss_hist = []
    for t in range(1, epochs + 1):
        caches = nn.forward(X)
        y_hat = caches[-1][1]
        loss = nn.compute_loss(y, y_hat)
        loss_hist.append(loss)
        wGrads, bGrads, _ = nn.backward(X, y, caches)
        for i, layer in enumerate(nn.layers):
            m_W[i] = beta1 * m_W[i] + (1 - beta1) * wGrads[i]
            m_b[i] = beta1 * m_b[i] + (1 - beta1) * bGrads[i]
            v_W[i] = beta2 * v_W[i] + (1 - beta2) * (wGrads[i]**2)
            v_b[i] = beta2 * v_b[i] + (1 - beta2) * (bGrads[i]**2)
            mW_hat = m_W[i] / (1 - beta1**t)
            mb_hat = m_b[i] / (1 - beta1**t)
            vW_hat = v_W[i] / (1 - beta2**t)
            vb_hat = v_b[i] / (1 - beta2**t)
            layer.weights -= lr * mW_hat / (np.sqrt(vW_hat) + eps)
            layer.biases -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
    return loss_hist

In [ ]:
# Train with same initialization (seed=42) for fair comparison
epochs = 200

nn_sgd = create_nn(seed=42)
nn_mom = create_nn(seed=42)
nn_adam = create_nn(seed=42)

loss_sgd = train_sgd(nn_sgd, X_moons, y_moons_2d, epochs, lr=0.05)
loss_mom = train_momentum(nn_mom, X_moons, y_moons_2d, epochs, lr=0.05, beta=0.9)
loss_adam = train_adam(nn_adam, X_moons, y_moons_2d, epochs, lr=0.01)

In [ ]:
# Loss curves: SGD vs Momentum vs Adam
plt.figure(figsize=(9, 5))
plt.plot(loss_sgd, 'r-', label='SGD (lr=0.05)', alpha=0.8)
plt.plot(loss_mom, 'c-', label='Momentum (lr=0.05, β=0.9)', alpha=0.8)
plt.plot(loss_adam, 'm-', label='Adam (lr=0.01)', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.title('Optimizer Comparison: Neural Network Training on Moons')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Reflection:** How do the optimizer differences you saw in 2D carry over to real training? Adam typically converges faster and more smoothly; SGD may oscillate more. Momentum often sits between them. The neural network loss landscape is high-dimensional and non-convex — the same principles (zig-zag, momentum smoothing, adaptive step sizes) apply, but now across thousands or millions of parameters.

---
## Summary

This laboratory has demonstrated:

| Concept | Key Takeaway |
|---------|--------------|
| **Optimization as movement** | Parameters move through loss space; gradient points uphill, we step downhill |
| **Learning rate** | Controls step size; too small = slow, too large = oscillation/divergence |
| **Curvature / ill-conditioning** | Elliptical valleys cause zig-zag; condition number measures asymmetry |
| **Momentum** | Accumulates velocity, smooths oscillations, faster along shallow directions |
| **Adaptive (RMSProp, Adam)** | Per-parameter learning rates; often faster convergence in difficult landscapes |
| **Saddle points** | Flat regions with mixed curvature; common in high dimensions; slow escape |
| **NN training** | Same principles apply; Adam often preferred in practice for stability and speed |

Optimization is a **physical process** in parameter space — understanding it visually builds intuition for hyperparameter tuning and debugging training.